In [1]:
!pip install -q gradio transformers torch black autopep8 reportlab requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.9/88.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 12.3 MB/s eta 0:00:00


In [2]:
import torch
import ast
import re
import os
import io
import textwrap
import tempfile
import requests
from datetime import datetime

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU — inference will be slower.")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [3]:
class ChatbotConfig:
    MODEL_NAME = "ibm-granite/granite-3.3-2b-instruct"
    DEFAULT_TEMP = 0.7
    DEFAULT_MAX_TOKENS = 256
    DEFAULT_TOP_P = 0.95
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Model  : {ChatbotConfig.MODEL_NAME}")
print(f"Device : {ChatbotConfig.DEVICE}")

Model  : ibm-granite/granite-3.3-2b-instruct
Device : cuda


In [ ]:
from transformers import pipeline

print("Loading IBM Granite model (this may take a few minutes)...")

model_pipe = pipeline(
    "text-generation",
    model=ChatbotConfig.MODEL_NAME,
    device=0 if ChatbotConfig.DEVICE == "cuda" else -1,
    torch_dtype=torch.float16 if ChatbotConfig.DEVICE == "cuda" else torch.float32,
)

print("✅ Model loaded successfully!")

Loading IBM Granite model (this may take a few minutes)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
def detect_python_code(text: str) -> bool:
    """Return True if the text appears to contain Python code."""
    patterns = [
        r'def\s+\w+\s*\(',          # function definition
        r'class\s+\w+',             # class definition
        r'import\s+\w+',            # import statement
        r'from\s+\w+\s+import',     # from-import
        r'if\s+.*:',                # if statement
        r'for\s+.*\s+in\s+',        # for loop
        r'while\s+.*:',             # while loop
        r'```python',               # explicit markdown code fence
        r'print\s*\(',              # print call
        r'return\s+',               # return statement
    ]
    return any(re.search(p, text) for p in patterns)


# ─────────────────────────────────────────────
# 2.2  Syntax Validation via AST
# ─────────────────────────────────────────────
def check_syntax(code: str):
    """Return (is_valid, error_message_or_None)."""
    try:
        ast.parse(code)
        return True, None
    except SyntaxError as e:
        return False, f"Syntax Error (line {e.lineno}): {e.msg}"
    except Exception as e:
        return False, str(e)


# ─────────────────────────────────────────────
# 2.3  Code Fixing Engine
# ─────────────────────────────────────────────
def fix_code_indentation(code: str) -> str:
    """Normalise indentation: replace tabs with 4 spaces, strip trailing whitespace."""
    lines = code.splitlines()
    fixed = [line.rstrip().replace('\t', '    ') for line in lines]
    return '\n'.join(fixed)


def fix_code_brackets(code: str) -> str:
    """Balance mismatched brackets/parens/braces by appending missing closers."""
    pairs = {'(': ')', '[': ']', '{': '}'}
    stack = []
    for ch in code:
        if ch in pairs:
            stack.append(pairs[ch])
        elif ch in pairs.values():
            if stack and stack[-1] == ch:
                stack.pop()
    # Append any missing closing brackets in reverse order
    return code + ''.join(reversed(stack))


def format_code(code: str) -> str:
    """Format code with Black; fall back to autopep8 on failure."""
    try:
        import black
        return black.format_str(code, mode=black.Mode())
    except Exception:
        pass
    try:
        import autopep8
        return autopep8.fix_code(code, options={'aggressive': 1})
    except Exception:
        return code  # Return as-is if both formatters fail


def fix_python_code(code: str) -> dict:
    """
    Orchestrate the full fix pipeline.
    Returns a dict with keys: original, fixed, syntax_valid, error, changes.
    """
    # Strip markdown fences if present
    code = re.sub(r'^```python\s*', '', code.strip(), flags=re.MULTILINE)
    code = re.sub(r'```$', '', code.strip(), flags=re.MULTILINE)
    code = code.strip()

    original = code
    changes = []

    # Step 1: Fix indentation
    fixed = fix_code_indentation(code)
    if fixed != code:
        changes.append("Normalised indentation (tabs → spaces)")
    code = fixed

    # Step 2: Balance brackets
    fixed = fix_code_brackets(code)
    if fixed != code:
        changes.append("Balanced unmatched brackets/parentheses")
    code = fixed

    # Step 3: Format with Black / autopep8
    formatted = format_code(code)
    if formatted.strip() != code.strip():
        changes.append("Applied PEP 8 formatting (Black/autopep8)")
    code = formatted

    # Step 4: Final syntax check
    is_valid, error = check_syntax(code)

    return {
        'original': original,
        'fixed': code,
        'syntax_valid': is_valid,
        'error': error,
        'changes': changes,
    }


# ─────────────────────────────────────────────
# 3.1  Conversation History
# ─────────────────────────────────────────────
conversation_history = []


def reset_history():
    global conversation_history
    conversation_history = []


# ─────────────────────────────────────────────
# 3.2  Response Generation
# ─────────────────────────────────────────────
def generate_response(user_message: str, temperature: float, max_tokens: int, top_p: float) -> str:
    """Send the conversation history to the model and return the assistant reply."""
    global conversation_history
    conversation_history.append({"role": "user", "content": user_message})
    try:
        output = model_pipe(
            conversation_history,
            max_new_tokens=int(max_tokens),
            temperature=temperature,
            do_sample=True,
            top_p=top_p,
        )
        # The pipeline returns a list; the last generated message is the assistant reply
        reply = output[0]["generated_text"][-1]["content"]
    except Exception as e:
        reply = f"⚠️ Model error: {e}"
    conversation_history.append({"role": "assistant", "content": reply})
    return reply


# ─────────────────────────────────────────────
# 4.2  Export Utilities
# ─────────────────────────────────────────────
def download_txt():
    """Export conversation history as a plain-text file."""
    if not conversation_history:
        return None, "No conversation to export."
    lines = [f"=== The Syntax Surgeon — Chat Export ===",
             f"Exported: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
             f"Messages: {len(conversation_history)}",
             "=" * 50, ""]
    for msg in conversation_history:
        role = msg["role"].upper()
        lines.append(f"[{role}]")
        lines.append(msg["content"])
        lines.append("")
    content = "\n".join(lines)
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".txt", mode="w", encoding="utf-8")
    tmp.write(content)
    tmp.close()
    return tmp.name, "✅ TXT export ready."


def download_pdf():
    """Export conversation history as a styled PDF using ReportLab."""
    if not conversation_history:
        return None, "No conversation to export."
    try:
        from reportlab.lib.pagesizes import letter
        from reportlab.lib import colors
        from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
        from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
        from reportlab.lib.units import inch

        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".pdf")
        doc = SimpleDocTemplate(tmp.name, pagesize=letter,
                                leftMargin=inch, rightMargin=inch,
                                topMargin=inch, bottomMargin=inch)
        styles = getSampleStyleSheet()
        user_style = ParagraphStyle('User', parent=styles['Normal'],
                                    textColor=colors.HexColor('#1a3c6e'),
                                    fontName='Helvetica-Bold', fontSize=10,
                                    spaceAfter=4)
        asst_style = ParagraphStyle('Asst', parent=styles['Normal'],
                                    textColor=colors.HexColor('#4b0082'),
                                    fontName='Helvetica', fontSize=10,
                                    spaceAfter=4)
        title_style = ParagraphStyle('Title', parent=styles['Title'],
                                     fontSize=16, spaceAfter=12)

        story = [
            Paragraph("🔬 The Syntax Surgeon — Chat Export", title_style),
            Paragraph(f"Exported: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Messages: {len(conversation_history)}",
                      styles['Normal']),
            Spacer(1, 0.2 * inch),
        ]
        for msg in conversation_history:
            role = msg['role'].upper()
            content = msg['content'].replace('<', '&lt;').replace('>', '&gt;')
            style = user_style if msg['role'] == 'user' else asst_style
            story.append(Paragraph(f"<b>[{role}]</b>", style))
            # Wrap long code lines
            for line in content.split('\n'):
                story.append(Paragraph(line if line.strip() else "&nbsp;", styles['Normal']))
            story.append(Spacer(1, 0.15 * inch))

        doc.build(story)
        tmp.close()
        return tmp.name, "✅ PDF export ready."
    except Exception as e:
        return None, f"❌ PDF error: {e}"


# ─────────────────────────────────────────────
# 4.3  Pastebin Share Link
# ─────────────────────────────────────────────
def generate_share_link():
    """Upload conversation to Pastebin and return the public URL."""
    if not conversation_history:
        return "No conversation to share."
    lines = ["The Syntax Surgeon — Shared Chat\n",
             f"Shared: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n",
             "=" * 50 + "\n"]
    for msg in conversation_history:
        lines.append(f"[{msg['role'].upper()}]\n{msg['content']}\n\n")
    paste_content = ''.join(lines)
    try:
        resp = requests.post(
            'https://pastebin.com/api/api_post.php',
            data={
                'api_dev_key': 'PASTE_YOUR_PASTEBIN_API_KEY_HERE',  # replace with your key
                'api_option': 'paste',
                'api_paste_code': paste_content,
                'api_paste_name': 'Syntax Surgeon Chat',
                'api_paste_expire_date': '1W',
                'api_paste_private': '0',
            },
            timeout=10
        )
        if resp.status_code == 200 and resp.text.startswith('https'):
            return f"✅ Share link: {resp.text.strip()}"
        return f"Pastebin response: {resp.text}"
    except Exception as e:
        return f"❌ Share error: {e}"


print("✅ All core functions defined.")

In [ ]:
def chat_handler(user_message, chat_history, temperature, max_tokens, top_p):
    """
    Main handler called when the user clicks Send.
    Returns updated chat_history and clears the input box.
    """
    if not user_message.strip():
        return chat_history, ""

    assistant_reply_parts = []

    # ── Code path ──────────────────────────────────────────────
    if detect_python_code(user_message):
        result = fix_python_code(user_message)

        if result['changes']:
            assistant_reply_parts.append("🔧 **Code fixes applied:**")
            for change in result['changes']:
                assistant_reply_parts.append(f"  • {change}")
            assistant_reply_parts.append("")

        if result['syntax_valid']:
            assistant_reply_parts.append("✅ Syntax is valid after fixing.")
        else:
            assistant_reply_parts.append(f"⚠️ Remaining issue: {result['error']}")

        assistant_reply_parts.append("\n**Fixed code:**")
        assistant_reply_parts.append(f"```python\n{result['fixed']}\n```")

        # Also ask the model to explain the fix
        explanation_prompt = (
            f"I fixed the following Python code. Briefly explain what it does:\n\n"
            f"```python\n{result['fixed']}\n```"
        )
        explanation = generate_response(explanation_prompt, temperature, max_tokens, top_p)
        assistant_reply_parts.append("\n**Explanation:**")
        assistant_reply_parts.append(explanation)

    # ── Conversational path ────────────────────────────────────
    else:
        reply = generate_response(user_message, temperature, int(max_tokens), top_p)
        assistant_reply_parts.append(reply)

    full_reply = '\n'.join(assistant_reply_parts)
    chat_history.append((user_message, full_reply))
    return chat_history, ""


def clear_chat():
    reset_history()
    return [], "Chat cleared ✅"


print("✅ Chat handler defined.")

In [ ]:
import gradio as gr

with gr.Blocks(title="The Syntax Surgeon", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🔬 The Syntax Surgeon
    **AI-Powered Code Fixer & Conversational Assistant** — IBM Granite 3.3 2B Instruct
    """)

    with gr.Row():
        # ── Left column: main chat ──────────────────────────────
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=420, label="Conversation", bubble_full_width=False)

            msg_input = gr.Textbox(
                lines=3,
                placeholder="Ask anything or paste Python code to fix...",
                label="Your message",
                show_label=False,
            )
            send_btn = gr.Button("Send 🚀", variant="primary")

            with gr.Accordion("⚙️ Model Parameters", open=False):
                temperature = gr.Slider(0.1, 1.0, value=ChatbotConfig.DEFAULT_TEMP,
                                        step=0.05, label="Temperature")
                max_tokens  = gr.Slider(50, 1024, value=ChatbotConfig.DEFAULT_MAX_TOKENS,
                                        step=50, label="Max Tokens")
                top_p       = gr.Slider(0.1, 1.0, value=ChatbotConfig.DEFAULT_TOP_P,
                                        step=0.05, label="Top-P")

        # ── Right column: controls ──────────────────────────────
        with gr.Column(scale=1):
            clear_btn  = gr.Button("🗑️ Clear Chat")
            dl_txt_btn = gr.Button("📄 Download TXT")
            dl_pdf_btn = gr.Button("📑 Download PDF")
            share_btn  = gr.Button("🔗 Share to Pastebin")

            status_box  = gr.Textbox(label="Status", interactive=False, lines=2)
            txt_file    = gr.File(label="TXT Download", visible=False)
            pdf_file    = gr.File(label="PDF Download", visible=False)

    # ── Event wiring ────────────────────────────────────────────

    # Send on button click
    send_btn.click(
        fn=chat_handler,
        inputs=[msg_input, chatbot, temperature, max_tokens, top_p],
        outputs=[chatbot, msg_input],
    )

    # Send on Enter key (shift+enter for newline)
    msg_input.submit(
        fn=chat_handler,
        inputs=[msg_input, chatbot, temperature, max_tokens, top_p],
        outputs=[chatbot, msg_input],
    )

    # Clear chat
    clear_btn.click(fn=clear_chat, outputs=[chatbot, status_box])

    # Download TXT
    def handle_txt():
        path, msg = download_txt()
        if path:
            return gr.File(value=path, visible=True), msg
        return gr.File(visible=False), msg

    dl_txt_btn.click(fn=handle_txt, outputs=[txt_file, status_box])

    # Download PDF
    def handle_pdf():
        path, msg = download_pdf()
        if path:
            return gr.File(value=path, visible=True), msg
        return gr.File(visible=False), msg

    dl_pdf_btn.click(fn=handle_pdf, outputs=[pdf_file, status_box])

    # Share to Pastebin
    share_btn.click(fn=generate_share_link, outputs=status_box)


demo.launch(share=True)